In [0]:
%python
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils.utils_merge_into_tables import upsert_data

In [0]:
%python
catalog_olist = dbutils.widgets.get("catalog_olist")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")
table_silver = dbutils.widgets.get("table_silver")
output_table = dbutils.widgets.get("output_table")
sk_table_olist = dbutils.widgets.get('sk_table')

In [0]:
CREATE OR REPLACE TEMPORARY VIEW silver_table_products AS
SELECT
*
FROM ${catalog}.${schema_silver}.${table_silver};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ${output_table} AS
SELECT
XXHASH64(product_id) AS SK_PRODUCT,
product_id AS ID_PRODUCT,
COALESCE(REGEXP_REPLACE(UPPER(product_category_name), '_', ' '),'CATEGORIA NAO INFORMADA') AS NM_CATEGORY,
product_weight_g AS WEIGHT_G,
product_length_cm AS LENGTH_CM,
product_height_cm AS HEIGHT_CM,
product_width_cm AS WIDTH_CM,
(product_length_cm * product_height_cm * product_width_cm) AS VOLUME_CM3
FROM silver_table_products;

## Merge Table

In [0]:
%run ../setup/00_aws_connection

In [0]:
%python
df_dim_products = spark.table(output_table)
full_table_name =  f'{catalog_olist}.{schema_gold}.{output_table}'

upsert_data(df_dim_products, full_table_name,sk_table_olist,name_bucket,layer='gold')
